# Data Generation Open-Meteo

## Imports

In [1]:
import os
import requests
import pandas as pd
import pvlib

## Choose Location
- Source: [(https://earth.google.com/)]

Set the approximate latitude and longitude for Australian postcode 2076. This is the postcode used by the ausgrid dataset. If using smart-meter data from another location, change these parameters.

In [2]:
time_zone = 'Australia/Sydney'
lat = -33.8688
lon = 151.2093

## Load Data

### Directories

In [3]:
base_dir = os.path.join(os.getcwd(), '..')

processed_data_dir = os.path.join(base_dir, 'processed_data')
if not os.path.exists(processed_data_dir): os.makedirs(processed_data_dir)

## Load Data
- Source: [(https://open-meteo.com/en/docs)]
- Source: [(https://gu-eresearch.github.io/web_scraping_workshop/content/3-APIs.html)]

Open-Meteo combines weather model outputs from multiple national weather services. For each location, the highest-resoltion model is chosen automatically. The API endpoint `/v1/forecast` accepts a geographical coordinate, list of weather variables, and start and end date. Days always start at `0:00` and contains 168 hours. The parameter `hourly=` accepts a list of variables, for which data will be returned at an hourly time-resolution. The `minute_15=` parameter can be used to get 15-minute-resolution data. If only hourly-resolution models are available, then the data is automatically interpolated.

We could download the data as CSV's from the Open-Meteo website, but it is more convenient to use the API. In webscraping, an API allows python to access a database stored on a webpage. Using a website's API is often the easiest way to access its data. The URL contains the `endpoint` to the base webstie, and the `relative path` to the dataset of interest. We can pull the data into python using `requests.get()`.

In [4]:
variable_list = ['temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration']

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    'latitude': lat,
    'longitude': lon,
    'start_date': '2010-07-01',
    'end_date': '2013-06-30',
    'hourly': variable_list,
    'models': 'era5',
    'timezone': 'Australia/Sydney',
    'temperature_unit': 'celsius',
    'precipitation_unit': 'mm'}

response = requests.get(url, params=params)
response.raise_for_status()

df = pd.DataFrame(response.json()['hourly'])

We request hourly historical ERA5 weather data for that location from July 2010 through June 2013, including temperature wind, solar radiation, cloud cover, and sunshine duration. We checks that the request succeeded, then converts the returned hourly data into a pandas DataFrame. Next, we inspect the dataframe:

In [5]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Missing values: {df.isna().sum().sum()}\n")
display(df.head())

Shape: (26304, 8)
Columns: ['time', 'temperature_2m', 'wind_speed_10m', 'shortwave_radiation', 'cloud_cover', 'direct_normal_irradiance', 'diffuse_radiation', 'sunshine_duration']
Missing values: 0



,time,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover,direct_normal_irradiance,diffuse_radiation,sunshine_duration
0,2010-07-01T00:00,4.7,9.1,0.0,0,0.0,0.0,0.0
1,2010-07-01T01:00,4.3,8.3,0.0,1,0.0,0.0,0.0
2,2010-07-01T02:00,4.0,8.4,0.0,9,0.0,0.0,0.0
3,2010-07-01T03:00,2.9,8.5,0.0,22,0.0,0.0,0.0
4,2010-07-01T04:00,2.7,8.2,0.0,9,0.0,0.0,0.0


The dataset contains 26,304 hourly records from July 2010 through June 2013, with eight columns: the timestamp and seven weather variables. There are no missing values. The first rows show nighttime readings, so zero solar radiation and sunshine are expected.

## Convert datetime columns

In [6]:
df['datetime'] = pd.to_datetime(df['time'])
df.drop(columns=['time'], inplace=True)

### Interpolate datetimes

The Open-Meteo time resolution is one hour, where the Ausgrid time resolution is thirty minutes. If using smart-meter data with a different time-resolution, this code will need to be updated.

In [7]:
df = (
    df.set_index("datetime")
      .resample("30min")
      .interpolate()
      .reset_index())

## Calculate sun position
- Source: [(https://pvlib-python.readthedocs.io/en/stable/)]

The `pvlib` library provides functions and classes for simulating the performance of photovoltaic energy systems. For now, we are only using the library to calculate solar position.

Solar position can help predict solar generation because the sun’s angle affects how much irradiance reaches the panels. We use the python library `pvlib` calculates the sun's zenith angle (height in the sky) and azimuth (compass direction) at each timestamp, using the site's coordinates.

In [8]:
local_times = pd.DatetimeIndex(df['datetime']).tz_localize(
    'Australia/Sydney', ambiguous='NaT', nonexistent='shift_forward')

solar_position = pvlib.solarposition.get_solarposition(
    time=local_times, latitude=lat, longitude=lon)

sun = pd.DataFrame({
    'datetime': df['datetime'],
    'solar_zenith': solar_position['zenith'].values,
    'solar_azimuth': solar_position['azimuth'].values})

display(sun.head(10))

,datetime,solar_zenith,solar_azimuth
0,2010-07-01 00:00:00,169.282674,178.549344
1,2010-07-01 00:30:00,167.294464,145.473811
2,2010-07-01 01:00:00,162.849249,124.686406
3,2010-07-01 01:30:00,157.361912,112.286882
4,2010-07-01 02:00:00,151.442416,104.049872
5,2010-07-01 02:30:00,145.330051,97.938249
6,2010-07-01 03:00:00,139.132636,92.991072
7,2010-07-01 03:30:00,132.907799,88.713492
8,2010-07-01 04:00:00,126.692106,84.824391
9,2010-07-01 04:30:00,120.512951,81.148108


Finally, we append the sun position to the weather dataframe.

In [9]:
df = df.merge(sun, on='datetime', how='left')

## Save data

In [10]:
save_filepath = os.path.join(processed_data_dir, 'open_meteo_processed_data.csv')
df.to_csv(save_filepath, index=False)